## Imports

In [1]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please u

In [2]:
from funcoes_escoragem import *

## Diretório

In [3]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [4]:
project_id = 'loft-dl-fintech'

# Parte 1

### CredPago - Consulta Realizada**

In [5]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 4 WEEK)
    AND DATE(data) < CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,45030551867,4107788,2026-06-17,2026-06-17,1,NI,11439.500000000,BLEND3_3,4107788,"{""pessoas"":[{""nome"":""ISABELLA CAMILA RODRIGUES...",...,31,1778785248777,"{""imobiliaria_id"":""17831"",""imovel"":{""tipo"":""CO...",2026-06-17 18:00:32+00:00,1781719230023371406,45030551867,A,2026-06-17 18:02:12.053000+00:00,2026-06-17 13:27:59+00:00,APROVAR
1,4193086950,4173906,2026-06-05,2026-06-05,1,NI,5137.500000000,BLEND3_3,4173906,"{""pessoas"":[{""nome"":""MARCIO JOSE MIRANDA PINTO...",...,34,1778785248777,"{""valor_aluguel"":""2450"",""imobiliaria_id"":5910,...",2026-06-05 18:00:26+00:00,1780682424622120599,04193086950,C,2026-06-05 18:09:02.787000+00:00,2026-06-05 14:37:25+00:00,APROVAR
2,51646049810,4178869,2026-06-23,2026-06-23,1,C,1644.000000000,BLEND_REGRESSAO_2026,4178869,"{""pessoas"":[{""nome"":""PEDRO HENRIQUE FERNANDES ...",...,34,1778785248777,"{""valor_aluguel"":""850"",""imobiliaria_id"":3780,""...",2026-06-24 08:00:33+00:00,1782288031800500566,51646049810,C,2026-06-24 08:06:53.124000+00:00,2026-06-23 17:07:35+00:00,DERIVAR
3,3676542029,4179100,2026-06-10,2026-06-10,1,E,1918.000000000,BLEND3_3,4179100,"{""pessoas"":[{""nome"":""PABLO GUILHERMO GRAZIADEI...",...,34,1778785248777,"{""valor_aluguel"":""1200"",""imobiliaria_id"":2687,...",2026-06-10 18:00:26+00:00,1781114422138134428,03676542029,C,2026-06-10 18:09:13.300000+00:00,2026-06-10 10:46:47+00:00,APROVAR
4,12742656650,4181824,2026-06-12,2026-06-12,2,A,4110.000000000,HVA3,4181824,"{""pessoas"":[{""nome"":""GUSTAVO DE ALMEIDA MONTEI...",...,34,1778785248777,"{""valor_aluguel"":""1500"",""imobiliaria_id"":35785...",2026-06-12 18:00:36+00:00,1781287234044418877,12742656650,C,2026-06-12 18:05:20.330000+00:00,2026-06-12 14:41:06+00:00,APROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108643,9701118600,4321799,2026-07-01,2026-07-01,2,NI,7055.500000000,HVA3,4321799,"{""pessoas"":[{""nome"":""FERNANDA VALERIO DE ALMEI...",...,36,1778785248777,"{""valor_aluguel"":1349.56,""matchmaking_on"":fals...",2026-07-02 08:01:26+00:00,1782979285357951647,09701118600,D,2026-07-02 08:12:00.456000+00:00,2026-07-01 18:24:33+00:00,APROVAR
108644,4240289300,4321808,2026-07-01,2026-07-01,1,NI,2055.000000000,BLEND3_3,4321808,"{""pessoas"":[{""nome"":""CHARLES BEZERRA LAGO"",""do...",...,32,1778785248777,"{""valor_aluguel"":1200,""matchmaking_on"":false,""...",2026-07-02 08:01:26+00:00,1782979285472415309,04240289300,B,2026-07-02 08:12:00.884000+00:00,2026-07-01 18:31:09+00:00,APROVAR
108645,99359820091,4321857,2026-07-01,2026-07-01,1,A,3630.500000000,BLEND3_3,4321857,"{""pessoas"":[{""nome"":""FABIANA DE MELO NEVES SIL...",...,33,1778785248777,"{""valor_aluguel"":1170,""matchmaking_on"":false,""...",2026-07-02 08:01:48+00:00,1782979307177311781,99359820091,E,2026-07-02 08:12:28.844000+00:00,2026-07-01 19:15:04+00:00,REPROVAR
108646,3687612074,4321900,2026-07-01,2026-07-01,1,NI,2055.000000000,BLEND3_3,4321900,"{""pessoas"":[{""nome"":""CARMINE LUISA WAGNER"",""do...",...,33,1778785248777,"{""valor_aluguel"":4500,""matchmaking_on"":false,""...",2026-07-02 08:02:48+00:00,1782979366994111270,03687612074,E,2026-07-02 08:13:36.527000+00:00,2026-07-01 20:06:42+00:00,REPROVAR


In [6]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

In [7]:
credpago_df['model'].value_counts(dropna=False)

model
BLEND3_3                93190
BLEND_REGRESSAO_2026     8766
HVA3                     4272
BVS_CUSTOM               1737
HVA4                      672
                           11
Name: count, dtype: int64

## Multiproponente

In [8]:
credpago_df['qtd_proponentes'].value_counts(dropna=False)

qtd_proponentes
1    104468
2      3864
3       279
4        35
6         1
5         1
Name: count, dtype: Int64

In [9]:
credpago_df['qtd_proponentes'].value_counts(normalize=True, dropna=False)

qtd_proponentes
1    0.961527
2    0.035564
3    0.002568
4    0.000322
6    0.000009
5    0.000009
Name: proportion, dtype: Float64

## Quebrar JSON

In [10]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [11]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

contrato_df = (
    pd.json_normalize(payload_parsed, sep="_")
    .add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.
)

pessoa_df = pd.json_normalize(
    payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0]),
    sep="_",
).add_prefix("pessoa_")

In [12]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        for pend in p.get("anotacoesFinanceirasSerasa", {}).get("PendenciasPagamentoPF", []):
            pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna())))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

In [13]:
credpago_expandido = (
    credpago_df.loc[valid]
    .join(contrato_df, how="left")   # contrato_df index = parsed.dropna().index
    .join(pessoa_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)

In [14]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])

### CredPago - Imóvel e Histórico de Valor

In [15]:
query = """
WITH property_type AS (
  SELECT
    id AS contract_id,
    CASE WHEN tp_imovel = 2 THEN 0 ELSE 1 END AS property_type,
    CAST(id_cidade_ibge AS INT64) AS id_cidade_ibge,    --nova


    TRIM(
    REGEXP_REPLACE(
        REGEXP_REPLACE(
        UPPER(
            REGEXP_REPLACE(
            NORMALIZE(COALESCE(cidade, ''), NFD),
            r'\pM', ''                      -- tira acento (marcas)
            )
        ),
        r'[^A-Z0-9 ]', ' '                 -- tira lixo (agora já está tudo em maiúsculo)
        ),
        r'\s+', ' '                          -- colapsa espaços
    )
    ) AS contract_city_nm, --nova

    CAST(id_imobiliaria AS INT64) AS agency_id,  --nova
    DATE(criado_em) AS requested_at
  FROM `loft-dl-fintech.bronze_credpago.imovel`
  WHERE DATE(criado_em) >= DATE('2026-01-01')
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY criado_em DESC) = 1
),

rental_value AS (
  SELECT
    id_imovel AS contract_id,
    CAST(vl_aluguel AS FLOAT64) AS rental_value,
    DATE(data) AS requested_at
  FROM `loft-dl-fintech.bronze_credpago.historico_valor`
  WHERE DATE(data) >= DATE('2026-01-01')
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id_imovel ORDER BY data DESC) = 1
)

SELECT
  COALESCE(p.contract_id, r.contract_id) AS contract_id,
  GREATEST(p.requested_at, r.requested_at) AS requested_at,
  p.property_type,
  p.id_cidade_ibge,
  p.contract_city_nm,
  p.agency_id,
  r.rental_value
FROM property_type p
FULL OUTER JOIN rental_value r
  ON p.contract_id = r.contract_id;
"""

credpago_imovel_df = pd.read_gbq(query, project_id=project_id)

In [16]:
credpago_imovel_df.head()

,contract_id,requested_at,property_type,id_cidade_ibge,contract_city_nm,agency_id,rental_value
0,3568933,2026-01-01,1,3507506,BOTUCATU,23453,1350.00
1,3569117,2026-01-02,1,4314100,PASSO FUNDO,1145,1490.00
2,3569170,2026-01-02,1,4212809,BALNEARIO PICARRAS,40682,2800.00
3,3569176,2026-01-02,1,3511102,CATANDUVA,48852,850.00
4,3569187,2026-01-02,1,1100049,CACOAL,16921,1111.11


### Merge 

In [17]:
cp_final_df = pd.merge(credpago_clean, credpago_imovel_df.drop(columns=['requested_at']), on='contract_id', how='left')
cp_final_df.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,request,...,pessoa_ratings_HVA3,pessoa_ratings_BLEND_REGRESSAO_2026,qtde_pefin,valor_pefin_total,modalidades_pefin,property_type,id_cidade_ibge,contract_city_nm,agency_id,rental_value
0,45030551867,4107788,2026-06-17,2026-06-17,1,NI,BLEND3_3,5961723,31,"{""imobiliaria_id"":""17831"",""imovel"":{""tipo"":""CO...",...,NaN,NaN,NaN,NaN,NaN,0,3513801,DIADEMA,17831,6000.0
1,4193086950,4173906,2026-06-05,2026-06-05,1,NI,BLEND3_3,5903264,34,"{""valor_aluguel"":""2450"",""imobiliaria_id"":5910,...",...,NaN,NaN,NaN,NaN,NaN,1,4115200,MARINGA,5910,2450.0
2,51646049810,4178869,2026-06-23,2026-06-23,1,C,BLEND_REGRESSAO_2026,5993594,34,"{""valor_aluguel"":""850"",""imobiliaria_id"":3780,""...",...,E,C,2.0,7979.0,OUTRAS OPER,1,3552809,TABOAO DA SERRA,3780,850.0
3,3676542029,4179100,2026-06-10,2026-06-10,1,E,BLEND3_3,5926712,34,"{""valor_aluguel"":""1200"",""imobiliaria_id"":2687,...",...,NaN,NaN,2.0,261.0,"CRED CARTAO,FINANCIAMENT",1,4314902,PORTO ALEGRE,2687,1200.0
4,12742656650,4181824,2026-06-12,2026-06-12,2,A,HVA3,5941095,34,"{""valor_aluguel"":""1500"",""imobiliaria_id"":35785...",...,A,NaN,NaN,NaN,NaN,1,3152501,POUSO ALEGRE,35785,1500.0


In [18]:
cols_ = ["property_type", "rental_value", "id_cidade_ibge", "agency_id"]

pct_nulls = (
    cp_final_df[cols_].isna().mean().mul(100).sort_values(ascending=False)
)
pct_nulls

property_type     0.008284
id_cidade_ibge    0.008284
agency_id         0.008284
rental_value      0.005522
dtype: float64

In [19]:
pct_nulls_por_requested_at = (
    cp_final_df.groupby("requested_at")[cols_]
    .apply(lambda x: x.isna().mean().mul(100))
    .sort_index()
)

pct_nulls_por_requested_at

,property_type,rental_value,id_cidade_ibge,agency_id
requested_at,,,,
2026-06-04,0.000000,0.000000,0.000000,0.000000
2026-06-05,0.000000,0.000000,0.000000,0.000000
2026-06-06,0.000000,0.000000,0.000000,0.000000
2026-06-07,0.000000,0.000000,0.000000,0.000000
2026-06-08,0.000000,0.000000,0.000000,0.000000
2026-06-09,0.012629,0.000000,0.012629,0.012629
2026-06-10,0.018083,0.018083,0.018083,0.018083
2026-06-11,0.020321,0.000000,0.020321,0.020321
2026-06-12,0.000000,0.000000,0.000000,0.000000


## Escoragem Blend3

In [20]:
rename_dict = {
    "message_scores_BVS_CUSTOM": "SCRCRDPNMGRLPFLGBCLFCREDPGV1",#
    "message_scores_HVA4": "SERASA_HVA4",
    "pessoa_idade": "age",
    "property_type": "property_type",  # peguei de fora da consulta realizada
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "rental_value": "rental_value",  # peguei de fora da consulta realizada
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [21]:
vars_blend4 = ['SCRCRDPNMGRLPFLGBCLFCREDPGV1', 'SERASA_HVA4', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [22]:
df_predict_blend3 = (cp_final_df.copy()).rename(columns=rename_dict)
# df_predict = df_predict[id_vars+vars_blend4]
df_predict_blend3["REGRA_BLEND_3"] = np.where(
    df_predict_blend3["SCRCRDPNMGRLPFLGBCLFCREDPGV1"] >= 435,
    "BLEND3",
    "E_BVS",
)

df_predict_blend3["SCORE_BVS"] = df_predict_blend3["SCRCRDPNMGRLPFLGBCLFCREDPGV1"]
df_predict_blend3["SCORE_SERASA"] = df_predict_blend3["SERASA_HVA4"]
df_predict_blend3["RENDA"] = df_predict_blend3["income"]

df_predict_blend3.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,request,...,modalidades_pefin,property_type,id_cidade_ibge,contract_city_nm,agency_id,rental_value,REGRA_BLEND_3,SCORE_BVS,SCORE_SERASA,RENDA
0,45030551867,4107788,2026-06-17,2026-06-17,1,NI,BLEND3_3,5961723,31,"{""imobiliaria_id"":""17831"",""imovel"":{""tipo"":""CO...",...,NaN,0,3513801,DIADEMA,17831,6000.0,BLEND3,874.0,644.0,11439.5
1,4193086950,4173906,2026-06-05,2026-06-05,1,NI,BLEND3_3,5903264,34,"{""valor_aluguel"":""2450"",""imobiliaria_id"":5910,...",...,NaN,1,4115200,MARINGA,5910,2450.0,BLEND3,643.0,592.0,5137.5
2,51646049810,4178869,2026-06-23,2026-06-23,1,C,BLEND_REGRESSAO_2026,5993594,34,"{""valor_aluguel"":""850"",""imobiliaria_id"":3780,""...",...,OUTRAS OPER,1,3552809,TABOAO DA SERRA,3780,850.0,BLEND3,654.0,NaN,1644.0
3,3676542029,4179100,2026-06-10,2026-06-10,1,E,BLEND3_3,5926712,34,"{""valor_aluguel"":""1200"",""imobiliaria_id"":2687,...",...,"CRED CARTAO,FINANCIAMENT",1,4314902,PORTO ALEGRE,2687,1200.0,BLEND3,666.0,369.0,1918.0
4,12742656650,4181824,2026-06-12,2026-06-12,2,A,HVA3,5941095,34,"{""valor_aluguel"":""1500"",""imobiliaria_id"":35785...",...,NaN,1,3152501,POUSO ALEGRE,35785,1500.0,E_BVS,NaN,NaN,4110.0


## Criação de Variáveis

In [23]:
df_predict_blend3_vars = prepare_blend3_variables(df_predict_blend3)
df_predict_blend3_escorada = predict_blend3(df_predict_blend3_vars)

## Rating

In [24]:
bvs = pd.to_numeric(df_predict_blend3_escorada["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(df_predict_blend3_escorada["predict_blend3_2_to_score"], errors="coerce")

conditions = [
    bvs.between(800, 1000),                                    # 800 ≤ BVS ≤ 1000  → A
    bvs.between(700, 799),                                     # 700 ≤ BVS < 800   → B
    bvs.between(620, 699),                                     # 620 ≤ BVS < 700   → C
    bvs.between(435, 619) & score.between(476, 1000),          # 435 ≤ BVS < 620, 476 ≤ pred ≤ 1000 → B
    bvs.between(435, 619) & score.between(399, 475),           # 435 ≤ BVS < 620, 399 ≤ pred < 476  → C
    bvs.between(435, 619) & score.between(387, 398),           # 435 ≤ BVS < 620, 387 ≤ pred < 399  → D
    bvs.between(435, 619) & score.between(0, 386),             # 435 ≤ BVS < 620, 0 ≤ pred < 387    → E
    bvs.between(0, 434),                                       # 0 ≤ BVS < 435    → E
]

choices = ["A", "B", "C", "B", "C", "D", "E", "E"]

df_predict_blend3_escorada["rating_manual_blend3"] = np.select(conditions, choices, default=None)

In [25]:
bvs = pd.to_numeric(df_predict_blend3_escorada["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(df_predict_blend3_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    bvs.between(800, 1000),                                    # 800 ≤ BVS ≤ 1000  → A
    bvs.between(700, 799),                                     # 700 ≤ BVS < 800   → B
    bvs.between(620, 699),                                     # 620 ≤ BVS < 700   → C
    bvs.between(435, 619) & score.between(476, 1000),          # 435 ≤ BVS < 620, 476 ≤ pred ≤ 1000 → B
    bvs.between(435, 619) & score.between(399, 475),           # 435 ≤ BVS < 620, 399 ≤ pred < 476  → C
    bvs.between(435, 619) & score.between(387, 398),           # 435 ≤ BVS < 620, 387 ≤ pred < 399  → D
    bvs.between(435, 619) & score.between(0, 386),             # 435 ≤ BVS < 620, 0 ≤ pred < 387    → E
    bvs.between(0, 434),                                       # 0 ≤ BVS < 435    → E
]

choices = ["A", "B", "C", "B", "C", "D", "E", "E"]

df_predict_blend3_escorada["rating_json_blend3"] = np.select(conditions, choices, default=None)

## Salvar

In [26]:
df_predict_blend3_escorada.to_csv(ANALYTICS_DIR/"df_predict_blend3.csv", index=False)

## Escoragem Blend4

In [27]:
rename_dict = {
    "message_scores_BVS_CUSTOM": "score_proposto__bvs",#
    "message_scores_HVA4": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "property_type": "property_type",  # peguei de fora da consulta realizada
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "rental_value": "rental_value",  # peguei de fora da consulta realizada
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [28]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [29]:
df_predict = (cp_final_df.copy()).rename(columns=rename_dict)
# df_predict = df_predict[id_vars+vars_blend4]
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] >= 334,
    "BLEND4",
    "E_BVS",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,request,...,modalidades_pefin,property_type,id_cidade_ibge,contract_city_nm,agency_id,rental_value,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,45030551867,4107788,2026-06-17,2026-06-17,1,NI,BLEND3_3,5961723,31,"{""imobiliaria_id"":""17831"",""imovel"":{""tipo"":""CO...",...,NaN,0,3513801,DIADEMA,17831,6000.0,BLEND4,874.0,644.0,11439.5
1,4193086950,4173906,2026-06-05,2026-06-05,1,NI,BLEND3_3,5903264,34,"{""valor_aluguel"":""2450"",""imobiliaria_id"":5910,...",...,NaN,1,4115200,MARINGA,5910,2450.0,BLEND4,643.0,592.0,5137.5
2,51646049810,4178869,2026-06-23,2026-06-23,1,C,BLEND_REGRESSAO_2026,5993594,34,"{""valor_aluguel"":""850"",""imobiliaria_id"":3780,""...",...,OUTRAS OPER,1,3552809,TABOAO DA SERRA,3780,850.0,BLEND4,654.0,NaN,1644.0
3,3676542029,4179100,2026-06-10,2026-06-10,1,E,BLEND3_3,5926712,34,"{""valor_aluguel"":""1200"",""imobiliaria_id"":2687,...",...,"CRED CARTAO,FINANCIAMENT",1,4314902,PORTO ALEGRE,2687,1200.0,BLEND4,666.0,369.0,1918.0
4,12742656650,4181824,2026-06-12,2026-06-12,2,A,HVA3,5941095,34,"{""valor_aluguel"":""1500"",""imobiliaria_id"":35785...",...,NaN,1,3152501,POUSO ALEGRE,35785,1500.0,E_BVS,NaN,NaN,4110.0


In [30]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [31]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [32]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [33]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [34]:
df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,request,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,45030551867,4107788,2026-06-17,2026-06-17,1,NI,BLEND3_3,5961723,31,"{""imobiliaria_id"":""17831"",""imovel"":{""tipo"":""CO...",...,0.513120,-0.40,0.0,0.208859,0.00000,0.000000,0.207235,793.0,A,A
1,4193086950,4173906,2026-06-05,2026-06-05,1,NI,BLEND3_3,5903264,34,"{""valor_aluguel"":""2450"",""imobiliaria_id"":5910,...",...,0.361516,0.55,0.0,0.103687,0.00000,-0.615123,0.336825,663.0,A,A
2,51646049810,4178869,2026-06-23,2026-06-23,1,C,BLEND_REGRESSAO_2026,5993594,34,"{""valor_aluguel"":""850"",""imobiliaria_id"":3780,""...",...,0.000000,-0.45,9.0,0.192366,0.00000,0.000000,0.447844,552.0,A,E
3,3676542029,4179100,2026-06-10,2026-06-10,1,E,BLEND3_3,5926712,34,"{""valor_aluguel"":""1200"",""imobiliaria_id"":2687,...",...,-0.288630,-0.15,2.0,0.432296,-0.02735,-0.424435,0.475063,525.0,A,D
4,12742656650,4181824,2026-06-12,2026-06-12,2,A,HVA3,5941095,34,"{""valor_aluguel"":""1500"",""imobiliaria_id"":35785...",...,0.000000,-0.35,0.0,-0.143537,0.00000,0.000000,0.487683,512.0,A,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108642,65134435987,4321542,2026-07-01,2026-07-01,1,NI,BLEND3_3,6032915,34,"{""valor_aluguel"":1200,""matchmaking_on"":false,""...",...,-0.344023,1.20,0.0,0.892963,0.00000,-0.080405,0.469700,530.0,A,E
108644,4240289300,4321808,2026-07-01,2026-07-01,1,NI,BLEND3_3,6033297,32,"{""valor_aluguel"":1200,""matchmaking_on"":false,""...",...,0.440233,0.50,0.0,0.340163,0.00000,-0.194504,0.404608,595.0,A,B
108645,99359820091,4321857,2026-07-01,2026-07-01,1,A,BLEND3_3,6033357,33,"{""valor_aluguel"":1170,""matchmaking_on"":false,""...",...,-0.481050,0.60,5.0,-0.237843,0.00000,1.628419,0.667621,332.0,E,E
108646,3687612074,4321900,2026-07-01,2026-07-01,1,NI,BLEND3_3,6033412,33,"{""valor_aluguel"":4500,""matchmaking_on"":false,""...",...,-0.282799,-0.50,0.0,3.887296,0.00000,-0.397626,0.565779,434.0,C,E


In [35]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [36]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)

# Parte 2

## Dados do Funil

In [37]:
query = '''
WITH tb_leads AS (
  SELECT 
    rf.contract_id,
    date(rf.dt_lead) as dt_lead,
    date(cf.requested_at) as requested_at,
    date(rf.dt_proposta_iniciada) as iniciada_at,
    date(rf.dt_proposta_enviada) as enviada_at,
    date(rf.activated_at) as activated_at,
    date(rf.cancelled_at) as cancelled_at,
    date(rf.dt_saida) as dt_saida,
    cf.tipo_contrato,
    rd.product_nm,
    cf.bureau_nm,
    CASE 
      WHEN cf.bureau_nm = 'BLEND_REGRESSAO_2026' AND CAST(ca.bvs_cust_score_nr AS FLOAT64) >= 413 THEN 'BLEND_REGRESSAO_2026'
      WHEN cf.bureau_nm = 'BVS_CUSTOM' AND CAST(ca.bvs_cust_score_nr AS FLOAT64) < 413 THEN 'BLEND_REGRESSAO_2026'
      ELSE COALESCE(cf.bureau_nm, '-1') END AS bureau_nm_ajust,
    cf.modelo_blend,
    cast(rf.total_rental_value_informed_nr as FLOAT64) as total_rental_value_informed_nr,
    cast(rf.rental_value_nr as FLOAT64) as rental_value_nr,
    -- cf.agency_id,
    -- cf.contract_city_nm,
    -- cf.person_age,
    cf.qtd_proponentes,
    cf.score_imobiliaria,
    cf.person_restriction_total_value,
    ca.bvs_cust_score_nr,
    ca.blend_regressao_predict_nr,
    cf.rating_score_ds,
    rd.pre_analysis_result,
    rd.lead_elegivel,
    rd.proposta_iniciada,
    rd.proposta_enviada,
    rd.proposta_aprovada,
    rd.proposta_ativada,
    rd.is_activeted
  FROM loft-dl-fintech.cp_gold.requests_fact AS rf
  LEFT JOIN loft-dl-fintech.cp_gold.requests_dim AS rd
    ON rf.contract_id = rd.contract_id
  LEFT JOIN loft-dl-fintech.cp_gold.credit_fact AS cf
    ON rf.contract_id = cf.contract_id
  LEFT JOIN loft-dl-fintech.cp_silver.int_credit_analyses AS ca
    ON rf.contract_id = ca.contract_id
  WHERE cf.tipo_contrato = 'PF'
),
first_defaults AS (
  SELECT
    contract_id
    , DATE(MIN(pendency_created_at)) AS first_comunicacao_date
    , DATE(MIN(pendency_at)) AS first_competencia_date
  FROM loft-dl-fintech.cp_gold.watchlist_fact
  WHERE pendency_type = "Inadimplência"
  GROUP BY contract_id
),
consulta_realizada AS(
  SELECT 
    CAST(id_externo AS INT64) AS contract_id,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.rendaConsiderada') AS FLOAT64) AS rendaConsiderada,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.rendaConsideradaContrato') AS FLOAT64) AS rendaConsideradaContrato,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.rendaUtilizada') AS rendaUtilizada,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.qtdeRestricoesContrato') AS FLOAT64) AS qtdeRestricoesContrato,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.valorRestricaoTotal') AS FLOAT64) AS valorRestricaoTotal,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.blend3Variables.hasPreviousContracts') AS blend32_hasPreviousContracts,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.blend3Variables.hadOverdueInvoiceInPreviousContracts') AS blend32_hadOverdueInvoiceInPreviousContracts,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.blend3Variables.cityPc4HistoryOver100Contracts') AS FLOAT64) AS blend32_cityPc4HistoryOver100Contracts,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.blend3Variables.realEstatePc4HistoryOver100Contracts') AS FLOAT64) AS blend32_realEstatePc4HistoryOver100Contracts,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.scores.BVS_CUSTOM') AS FLOAT64) AS bvs_custom_score,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.scores.HVA3') AS FLOAT64) AS hva3_score,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.scores.HVA4') AS FLOAT64) AS hva4_score,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.scores.BLEND_REGRESSAO_2026') AS FLOAT64) AS blend_regressao_2026_score,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.scores.BLEND3_2') AS FLOAT64) AS blend3_2_score,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.blendRegressaoPredict') AS FLOAT64) AS blend_regressao_predict,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.ratings.BVS_CUSTOM') AS bvs_custom_rating,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.ratings.HVA3') AS hva3_rating,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.ratings.HVA4') AS hva4_rating,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.ratings.BLEND_REGRESSAO_2026') AS blend_regressao_2026_rating,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.ratings.BLEND3_2') AS blend3_2_rating,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.errosTecnicos.0') as errosTecnicos,
    letra,
    ROW_NUMBER() OVER (PARTITION BY CAST(id_externo AS INT64) ORDER BY data DESC) as rn
  FROM `loft-dl-fintech.bronze_credpago.consulta_realizada` cr
  where REGEXP_CONTAINS(id_externo, r'^[0-9]+$')
)
SELECT
  l.*
  , case when bureau_nm_ajust = 'BLEND_REGRESSAO' then 
        case when bvs_cust_score_nr >= 800 then 'A'
            when bvs_cust_score_nr >= 700 then 'B'
            when bvs_cust_score_nr >= 625 then 'C'
            when (bvs_cust_score_nr >= 478 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr <= 0.12) then 'B'
            when (bvs_cust_score_nr >= 478 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr <= 0.17) then 'C'
            when (bvs_cust_score_nr >= 478 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr <= 0.271) then 'D'
            when (bvs_cust_score_nr >= 478 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr > 0.271) then 'E'
            when bvs_cust_score_nr < 478 then 'E' else '-1' end
      when bureau_nm_ajust in ('BLEND_REGRESSAO_2026') then
        case when bvs_cust_score_nr >= 800 then 'A'
            when bvs_cust_score_nr >= 700 then 'B'
            when bvs_cust_score_nr >= 625 then 'C'
            when (bvs_cust_score_nr >= 413 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr <= 0.132) then 'B'
            when (bvs_cust_score_nr >= 413 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr <= 0.184) then 'C'
            when (bvs_cust_score_nr >= 413 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr <= 0.271) then 'D'
            when (bvs_cust_score_nr >= 413 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr > 0.271) then 'E'
            when bvs_cust_score_nr < 413 then 'E' else '-1' end
      when bureau_nm_ajust = 'BLEND3_2' then 
        case when bvs_cust_score_nr >= 800 then 'A'
            when bvs_cust_score_nr >= 745 then 'B'
            when bvs_cust_score_nr >= 660 then 'C'
            when (bvs_cust_score_nr >= 470 and bvs_cust_score_nr < 660 and blend_regressao_predict_nr >= 470) then 'B'
            when (bvs_cust_score_nr >= 470 and bvs_cust_score_nr < 660 and blend_regressao_predict_nr >= 395) then 'C'
            when (bvs_cust_score_nr >= 470 and bvs_cust_score_nr < 660 and blend_regressao_predict_nr >= 366) then 'D'
            when (bvs_cust_score_nr >= 470 and bvs_cust_score_nr < 660 and blend_regressao_predict_nr < 366) then 'E'
            when bvs_cust_score_nr < 470 then 'E' else '-1' end
      when bureau_nm_ajust = 'BLEND3_3' then 
        case when bvs_cust_score_nr >= 800 then 'A'
            when date(requested_at) >= date('2026-04-20') AND bvs_cust_score_nr >= 700 THEN 'B'
            when bvs_cust_score_nr >= 750 then 'B'
            when bvs_cust_score_nr >= 620 then 'C'
            when (bvs_cust_score_nr >= 435 and bvs_cust_score_nr < 620 and blend_regressao_predict_nr >= 476) then 'B'
            when (bvs_cust_score_nr >= 435 and bvs_cust_score_nr < 620 and blend_regressao_predict_nr >= 399) then 'C'
            when (bvs_cust_score_nr >= 435 and bvs_cust_score_nr < 620 and blend_regressao_predict_nr >= 387) then 'D'
            when (bvs_cust_score_nr >= 435 and bvs_cust_score_nr < 620 and blend_regressao_predict_nr < 387) then 'E'
            when bvs_cust_score_nr < 435 then 'E' else '-1' end
  else 'Outros' end as rating
  , fd.first_comunicacao_date
  , fd.first_competencia_date
  , cr.rendaConsiderada
  , cr.rendaConsideradaContrato
  , cr.rendaUtilizada
  , cr.qtdeRestricoesContrato
  , cr.valorRestricaoTotal
  , cr.blend32_hasPreviousContracts
  , cr.blend32_hadOverdueInvoiceInPreviousContracts
  , cr.blend32_cityPc4HistoryOver100Contracts
  , cr.blend32_realEstatePc4HistoryOver100Contracts
  , cr.bvs_custom_score
  , cr.hva3_score
  , cr.hva4_score
  , cr.blend_regressao_2026_score
  , cr.blend3_2_score
  , cr.blend_regressao_predict
  , cr.bvs_custom_rating
  , cr.hva3_rating
  , cr.hva4_rating
  , cr.blend_regressao_2026_rating
  , cr.blend3_2_rating
  , cr.errosTecnicos
  , cr.letra
FROM tb_leads AS l
LEFT JOIN first_defaults as fd
  on l.contract_id = fd.contract_id
  and l.activated_at is not null
LEFT JOIN consulta_realizada as cr
  on l.contract_id = cr.contract_id
  and cr.rn = 1
where 1=1
    and date(l.requested_at) >= date('2026-05-01')
    and date(l.requested_at) < date(current_date())
'''

In [38]:
df_funil = pandas_gbq.read_gbq(query, project_id=project_id)
df_funil

Downloading: 100%|██████████|


,contract_id,dt_lead,requested_at,iniciada_at,enviada_at,activated_at,cancelled_at,dt_saida,tipo_contrato,product_nm,...,blend_regressao_2026_score,blend3_2_score,blend_regressao_predict,bvs_custom_rating,hva3_rating,hva4_rating,blend_regressao_2026_rating,blend3_2_rating,errosTecnicos,letra
0,4093994,2026-05-05,2026-05-05,NaT,NaT,NaT,NaT,NaT,PF,None,...,577.0,NaN,0.2664,D,E,None,D,None,None,D
1,4092658,2026-05-05,2026-05-05,2026-05-05,NaT,NaT,2026-06-20,NaT,PF,Smart,...,NaN,NaN,319.0000,C,None,C,None,None,None,C
2,4095405,2026-05-05,2026-05-05,NaT,NaT,NaT,NaT,NaT,PF,None,...,NaN,NaN,561.0000,B,None,E,None,None,None,B
3,4091446,2026-05-05,2026-05-05,NaT,NaT,NaT,NaT,NaT,PF,None,...,643.0,NaN,0.1268,C,C,None,C,None,None,C
4,4095446,2026-05-05,2026-05-05,2026-05-05,NaT,NaT,2026-06-20,NaT,PF,Smart Plus,...,NaN,NaN,715.0000,A,None,C,None,None,None,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227687,4311741,2026-06-30,2026-06-30,2026-06-30,2026-06-30,2026-06-30,NaT,NaT,PF,Smart Plus,...,NaN,NaN,751.0000,B,None,A,None,None,None,B
227688,4315179,2026-06-30,2026-06-30,2026-06-30,2026-06-30,NaT,NaT,NaT,PF,Smart Plus,...,NaN,NaN,651.0000,C,None,C,None,None,None,C
227689,4312184,2026-06-30,2026-06-30,2026-06-30,2026-06-30,2026-07-01,NaT,NaT,PF,Smart Plus,...,NaN,NaN,463.0000,E,None,E,None,None,None,C
227690,4312497,2026-06-30,2026-06-30,2026-06-30,2026-06-30,2026-07-01,NaT,NaT,PF,Smart Plus,...,NaN,NaN,299.0000,E,None,C,None,None,None,E


In [39]:
bvs = pd.to_numeric(df_funil["bvs_cust_score_nr"], errors="coerce")
score = pd.to_numeric(df_funil["blend_regressao_predict_nr"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.BVS",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

df_funil["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

In [40]:
bvs = pd.to_numeric(df_funil["bvs_cust_score_nr"], errors="coerce")
score = pd.to_numeric(df_funil["blend_regressao_predict_nr"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "9.E.BVS",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "2.A",
    "3.B+",
    "3.B+",
    "4.B",
    "4.B",
    "5.C",
    "6.D+",
    "7.D",
    "7.D",
    "8.E",
    "8.E",
]

df_funil["rating_pol_blend4"] = np.select(conditions, choices, default=None)

In [41]:
df_funil.groupby("rating_pol_blend4", dropna=False).size()

rating_pol_blend4
1.A+        3910
2.A        18394
3.B+       22340
4.B        23251
5.C        11475
6.D+       11094
7.D        21500
8.E        67750
9.E.BVS     9968
NaN        38010
dtype: int64

quem é esse NaN?

In [42]:
df_funil_blend4 = df_funil[df_funil["bureau_nm_ajust"] == "BLEND3_3"].copy()
df_funil_blend4["bureau_nm_ajust"] = df_funil_blend4["bureau_nm_ajust"].replace("BLEND3_3", "BLEND4")

In [43]:
df_funil_blend4 = pd.concat([df_funil_blend4, df_funil])
df_funil_blend4.to_csv(ANALYTICS_DIR/"df_funil_blend4.csv", index=False)

In [44]:
df_funil.to_csv(ANALYTICS_DIR/"df_funil_blend3.csv", index=False)

credit_fact: 
    pre_analysis_result

requests_fact:
    pre_analysis_result/analysis_status

requests_dim:
    pre_analysis_result/elegivel_decisao

# Modelos

Verificar última e ir apendando. Sempre D-1 (Credpago). Não dá para fazer isso na GOLD.

Focar no Blend4
Consistência, sem muito histórico.

Mudar a query
Uma única linha que tenha batido no blend4 e verificá-la, independente se depois eu bater e rodar blend4
Tudo na consulta_realizada que for blend4.

Gold possui loucuras (virada do modelo).

Monitor do Modelo: Apenas consulta realizada

Monitor do Modelo com dados do Funil: Aí entram dados da gold.

#
Para cada modelo ver: proporção de result_pre_analise